# 03 Neural stream inventory and decoder manifest

This notebook bridges the gap between the trial catalog and the first decoding notebook.

## Purpose
1. Open every NWB file listed in the session index.
2. Inventory acquisition and processing objects across sessions.
3. Identify candidate neural streams for decoding.
4. Build a decoder manifest that maps each session to a likely neural source.
5. Save figures, tables, and metadata for the baseline decoder notebook.

## Inputs
- `/kaggle/working/tables_session_index/session_master_index.csv`
- `/kaggle/working/tables_session_index/decoder_trial_table.csv`
- Raw NWB files under:
  `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_stream_inventory/all_object_manifest.csv`
- `tables_stream_inventory/candidate_stream_manifest.csv`
- `tables_stream_inventory/session_decoder_stream_map.csv`
- `meta_stream_inventory/stream_inventory_report.json`
- `figures_stream_inventory/*.png`

In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from pynwb import NWBHDF5IO

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

## Paths and notebook dependencies

This notebook expects the outputs from `02_session_index_and_trial_catalog.ipynb` to already exist.